This feature is only available in `Standard`, `Premium` and `Enterprise` subscription packages.

In [1]:
from futureexpert import (ActualsCovsConfiguration,
                          CovariateRef,
                          ExpertClient,
                          ForecastingConfig,
                          MatcherConfig,
                          ReportConfig)
from futureexpert.pool import PoolCovDefinition

# Log in using your credentials or alternatively provide FUTURE_USER and FUTURE_PW via environment variables or .env file
client = ExpertClient()

INFO:futureexpert.expert_client:Successfully logged in to futureEXPERT.


# Getting to know _POOL_
The client allows you to get an overview of the _POOL_ covariates. Use the parameter `granularity` to filter for data that matches the time series you want to forecast. Using the search parameter allows to perform a full-text search on the data. In this example, we want to get every data on a daily granularity that contains the keyword "Wetterdaten" in any of its metadata.

In [2]:
pool_cov_overview = client.get_pool_cov_overview(granularity="Day", search="Wetterdaten")
pool_cov_overview.pool_cov_information.head(20)

,name,description,pool_cov_id
0,Temperatur (täglicher Minimalwert; Aachen-Orsb...,Historische Wetterdaten und Vorhersagen,834d3833-a10c-34c3-a2b2-5a33464bdbc3
1,Luftfeuchtigkeit (täglicher Mittelwert; Hambur...,Historische Wetterdaten und Vorhersagen,46525f7c-b6fe-3198-a580-c9fada159f01
2,Luftfeuchtigkeit (täglicher Mittelwert; Stuttg...,Historische Wetterdaten und Vorhersagen,341067e1-089b-3cd0-a5f9-e55f01cc86f4
3,Temperatur (täglicher Maximalwert; Hamburg-Fuh...,Historische Wetterdaten und Vorhersagen,0b2266f8-fe82-3431-8daf-68ce660641df
4,Windgeschwindigkeit (täglicher Mittelwert; Ber...,Historische Wetterdaten und Vorhersagen,8aba2bc0-dba4-3b46-9ba7-87882db158c5
5,Windgeschwindigkeit (täglicher Mittelwert; Ber...,Historische Wetterdaten und Vorhersagen,a4446982-0128-3806-ae9a-30a650592823
6,Temperatur (täglicher Minimalwert; München-Flu...,Historische Wetterdaten und Vorhersagen,000715ab-e0dc-3175-b18f-90b3f50ae69b
7,Temperatur (täglicher Minimalwert; Erfurt-Weimar),Historische Wetterdaten und Vorhersagen,6a234098-1df7-35a3-b839-0e28ac8e84d4
8,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e
9,Luftdruck (täglicher Mittelwert; Würzburg),Historische Wetterdaten und Vorhersagen,8038a276-60a2-38a7-9be1-3412bba54412


If this is not enough information about every indicator, you can inspect the detailed overview.

In [3]:
pool_cov_overview.detailed_pool_cov_information.head()

,name,description,unit,source,link,regions,sectors,keywords,includes_forecasts,pool_cov_id,last_updated,number_of_versions,versions,first_observation,final_observation,granularity
0,Temperatur (täglicher Minimalwert; Aachen-Orsb...,Historische Wetterdaten und Vorhersagen,°C,Deutscher Wetterdienst,https://wetterdienst.readthedocs.io/,[DE],"[Energie, Einzelhandel]","[weather, Temperatur]",True,834d3833-a10c-34c3-a2b2-5a33464bdbc3,2026-02-18T05:00:45,654,[{'version_id': '6a6820c0-fe7c-44e1-bc06-91c28...,2011-04-01T00:00:00,2026-02-28T00:00:00,Day
1,Luftfeuchtigkeit (täglicher Mittelwert; Hambur...,Historische Wetterdaten und Vorhersagen,%,Deutscher Wetterdienst & Open Weather,via https://wetterdienst.readthedocs.io/ & htt...,[DE],"[Energie, Einzelhandel]","[weather, Luftfeuchtigkeit]",True,46525f7c-b6fe-3198-a580-c9fada159f01,2024-08-29T05:00:48,328,[{'version_id': '66b9ad74-81a8-44fe-98c0-e9824...,2010-01-01T00:00:00,2024-09-05T00:00:00,Day
2,Luftfeuchtigkeit (täglicher Mittelwert; Stuttg...,Historische Wetterdaten und Vorhersagen,%,Deutscher Wetterdienst & Open Weather,via https://wetterdienst.readthedocs.io/ & htt...,[DE],"[Energie, Einzelhandel]","[weather, Luftfeuchtigkeit]",True,341067e1-089b-3cd0-a5f9-e55f01cc86f4,2024-08-29T05:01:25,26,[{'version_id': '9d5bf87b-36bc-434f-a8a2-ab13a...,2010-01-01T00:00:00,2024-09-05T00:00:00,Day
3,Temperatur (täglicher Maximalwert; Hamburg-Fuh...,Historische Wetterdaten und Vorhersagen,°C,Deutscher Wetterdienst,https://wetterdienst.readthedocs.io/,[DE],"[Energie, Einzelhandel]","[weather, Temperatur]",True,0b2266f8-fe82-3431-8daf-68ce660641df,2026-02-18T05:00:32,648,[{'version_id': '2a8ac063-291a-464c-bd6a-68193...,2010-01-01T00:00:00,2026-02-28T00:00:00,Day
4,Windgeschwindigkeit (täglicher Mittelwert; Ber...,Historische Wetterdaten und Vorhersagen,m/s,Deutscher Wetterdienst,https://wetterdienst.readthedocs.io/,[DE],"[Energie, Einzelhandel]","[weather, Windgeschwindigkeit]",True,8aba2bc0-dba4-3b46-9ba7-87882db158c5,2025-11-10T05:00:10,506,[{'version_id': 'da5a3780-7c46-46e2-8df4-93e88...,2025-11-10T00:00:00,2025-11-20T00:00:00,Day


You can now use the `query` method to further specify which data you want to use. In this example we are only looking at Temperature data from Würzburg (by selecting data with those two terms in its name).

In [4]:
selected_indicators = pool_cov_overview.query('name.str.contains("Würzburg") and name.str.contains("Temperatur")')
selected_indicators.pool_cov_information

,name,description,pool_cov_id
0,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e
1,Temperatur (täglicher Maximalwert; Würzburg),Historische Wetterdaten und Vorhersagen,1d994b16-4819-3b82-b0f8-dd65fc3ab694


# Creating pool cov definitions
Once you have selected the exact subset of covariates you want to use, you need to create a definition with which you can continue working.

In [5]:
pool_covariate_definitions = selected_indicators.create_pool_cov_definitions()
pool_covariate_definitions

[PoolCovDefinition(pool_cov_id='d4ee8ec6-15ae-3626-8eda-ab06a931314e', version_id=None),
 PoolCovDefinition(pool_cov_id='1d994b16-4819-3b82-b0f8-dd65fc3ab694', version_id=None)]

Alternatively, you can directly create a list of pool covariate definitions. This is helpful if you know the covariate IDs without exploring the data (e.g. from a previous session).

In [6]:
pool_covariate_definitions = [PoolCovDefinition(pool_cov_id='1d994b16-4819-3b82-b0f8-dd65fc3ab694'),
                              PoolCovDefinition(pool_cov_id='d4ee8ec6-15ae-3626-8eda-ab06a931314e')]
pool_covariate_definitions

[PoolCovDefinition(pool_cov_id='1d994b16-4819-3b82-b0f8-dd65fc3ab694', version_id=None),
 PoolCovDefinition(pool_cov_id='d4ee8ec6-15ae-3626-8eda-ab06a931314e', version_id=None)]

# Creating forecasts: Option 1 - using the pool covariate definitions
Once you have created the list of definitions, you can use that list to create forecasts.

In [7]:
covs_configuration = [CovariateRef(name=name, lag=3) for name in selected_indicators.pool_cov_information.name.to_list()]
actuals_cov_configuration = [ActualsCovsConfiguration(actuals_name="my actuals", covs_configurations=covs_configuration)]

report_config_from_overview = ReportConfig(title='Using the pool_overview.',
                                           forecasting=ForecastingConfig(fc_horizon=5),
                                           pool_covs=pool_covariate_definitions,
                                           covs_configuration=actuals_cov_configuration)


# Creating forecasts: Option 2 - create a cov version
You can use the function `check_in_pool_covs` to create a version from the list of definitions. See [forecasts_with_covariates](notebooks/forecast_with_covariates.ipynb) for information on how to use a `covs_version`.

Use this approach if you create multiple forecasts at once (same day or week) for different sets of time series that all use the same covariates. Do not use this approach if you want to create forecasts much later - versions are not persisted forever.

In [8]:
pool_version_information = client.check_in_pool_covs(requested_pool_covs=pool_covariate_definitions)
pool_covs_version = pool_version_information.version_id
pool_covs_version

INFO:futureexpert.expert_client:Creating time series using checkin-pool...
INFO:futureexpert.expert_client:Finished time series creation.


'69965f2e38e17253ebc01a5b'

# Working with indicator versions
All _POOL_ covariates are updated regularly. Every time there is a new update, a new _version_  of that covariate is created. Whenever you select an covariate but do not specify a version, the newest version is used. This should be the desired behaviour in most use cases. However, sometimes it is required to use an old version of an covariate (simulating different configurations, ...). If that is the case, use one of the following functions to specify which version to use.

Before you can specify versions, you can inspect information on all available versions of an covariate. Select one specific covariate in the detailed overview. The function `get_versions_of_indicator` allows you to get all available information about that version.

In [9]:
selected_indicators.get_versions_of_pool_cov(selected_indicators.detailed_pool_cov_information.loc[0,])

,name,description,pool_cov_id,version_id,reference_time,created_at,first_observation,final_observation
0,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,edad7892-de0f-47ac-8427-5b62c55035c1,2026-02-18T05:00:39.494000,2026-02-18T05:00:41,2010-01-01T00:00:00,2026-02-28T00:00:00
1,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,cfaca94f-134a-4d2a-a9e3-166fd06227db,2026-02-17T05:02:18.906000,2026-02-17T05:02:20,2010-01-01T00:00:00,2026-02-27T00:00:00
2,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,2d754096-db23-4616-860d-08d2ff083112,2026-02-16T05:00:54.150000,2026-02-16T05:00:55,2010-01-01T00:00:00,2026-02-26T00:00:00
3,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,117bd51f-ef63-4306-8ff7-6bfccef3882f,2026-02-15T05:01:03.255000,2026-02-15T05:01:05,2010-01-01T00:00:00,2026-02-25T00:00:00
4,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,eb39430d-7a89-4502-a6f0-402d0babb5ea,2026-02-14T05:01:48.422000,2026-02-14T05:01:50,2010-01-01T00:00:00,2026-02-24T00:00:00
...,...,...,...,...,...,...,...,...
645,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,844a8116-513a-4ffe-acd1-1bfcf4bc787f,2023-09-09T05:03:07.411000,2023-09-09T05:03:44,2000-01-01T00:00:00,2023-09-16T00:00:00
646,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,b0f56da9-c15c-406c-b3ec-a5e529df0254,2023-09-08T12:44:33.145000,2023-09-08T12:45:07,2000-01-01T00:00:00,2023-09-15T00:00:00
647,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,6b074774-7606-413e-acce-c87a7ca40c85,2023-09-03T06:02:45.284000,2023-09-03T06:03:16,2000-01-01T00:00:00,2023-09-10T00:00:00
648,Temperatur (täglicher Minimalwert; Würzburg),Historische Wetterdaten und Vorhersagen,d4ee8ec6-15ae-3626-8eda-ab06a931314e,f0be38b9-0420-4567-9c8f-9a7288fdbcae,2023-08-27T06:02:40.389000,2023-08-27T06:03:09,2000-01-01T00:00:00,2023-09-03T00:00:00


# Specifying indicator versions

You can specify the desired version by adding it to the corresponding entry in the list of definitions.

In [10]:
pool_covariate_definitions[0].version_id = 'f2f814ba-03c7-47c6-9eb4-d7bbee95737f'
pool_covariate_definitions

[PoolCovDefinition(pool_cov_id='1d994b16-4819-3b82-b0f8-dd65fc3ab694', version_id='f2f814ba-03c7-47c6-9eb4-d7bbee95737f'),
 PoolCovDefinition(pool_cov_id='d4ee8ec6-15ae-3626-8eda-ab06a931314e', version_id=None)]

# Use the _MATCHER_ with _POOL_
All functionality described for creating forecasts with data from the _POOL_ also applies for using the _MATCHER_. Simply create a version and use that in the `MatcherConfig`. You can also directly pass the `pool_covariate_definitions`:

In [11]:
config_matcher = MatcherConfig(title='Covariate MATCHER',
                               actuals_version="actuals_version",
                               pool_covs=pool_covariate_definitions)